# Ćwiczenie 3 – Analiza trendu serii czasowej
**Stacja:** SULEJÓW &nbsp;|&nbsp; **Dane:** 1966–2017 (lata z pełnymi 12 miesiącami)

## 1. Wczytanie danych i obliczenie średnich rocznych temperatur

In [ ]:
import xlrd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

wb = xlrd.open_workbook('351190469_SULEJÓW.xls')
ws = wb.sheet_by_index(0)

rows = [ws.row_values(i) for i in range(1, ws.nrows)]
df = pd.DataFrame(rows, columns=[
    'kod', 'stacja', 'rok', 'miesiac', 'dzien',
    'tmax', 'tmin', 'tsr', 'opad'
])
df['rok'] = df['rok'].astype(int)

# odfiltruj niepełne lata (plik urywa się w lutym 2018)
miesiace_na_rok = df.groupby('rok')['miesiac'].nunique()
pelne_lata = miesiace_na_rok[miesiace_na_rok == 12].index
df = df[df['rok'].isin(pelne_lata)]

# średnie roczne temperatury (yi)
roczne = df.groupby('rok')['tsr'].mean().reset_index()
roczne.columns = ['rok', 'yi']
print(f"Zakres danych: {roczne['rok'].min()} – {roczne['rok'].max()}  ({len(roczne)} lat)")
print(roczne)

## 2. Wyznaczenie współczynników trendu liniowego

Model: $y = a \cdot t + b$

$$a = \frac{\sum_{i=1}^{n}(t_i - \bar{t})(y_i - \bar{y})}{\sum_{i=1}^{n}(t_i - \bar{t})^2}, \qquad b = \bar{y} - a\bar{t}$$

gdzie $t_i$ – kolejny numer roku w serii (1, 2, 3, …), $y_i$ – średnia roczna temperatura.

In [ ]:
n = len(roczne)
roczne['ti'] = np.arange(1, n + 1)   # t_i = 1, 2, ..., n

t_mean = roczne['ti'].mean()          # t̄
y_mean = roczne['yi'].mean()          # ȳ

licznik   = ((roczne['ti'] - t_mean) * (roczne['yi'] - y_mean)).sum()
mianownik = ((roczne['ti'] - t_mean) ** 2).sum()

a = licznik / mianownik               # współczynnik kierunkowy
b = y_mean - a * t_mean               # wyraz wolny

print(f"t̄  = {t_mean:.2f}")
print(f"ȳ  = {y_mean:.4f} °C")
print(f"a  = {a:.6f} °C/rok  →  {a*10:.4f} °C/10 lat")
print(f"b  = {b:.4f} °C")

## 3. Prognoza temperatury na rok 2100

In [ ]:
rok_start = roczne['rok'].iloc[0]     # 1966
t_2100 = 2100 - rok_start + 1        # numer porządkowy roku 2100
y_2100 = a * t_2100 + b

print(f"t dla roku 2100 = {t_2100}")
print(f"Prognozowana średnia roczna temperatura w 2100: {y_2100:.2f} °C")
print(f"Wzrost względem średniej wieloletniej ({y_mean:.2f} °C): {y_2100 - y_mean:.2f} °C")

## 4. Średnia krocząca 30-letnia

$$\bar{y}_{i+n} = \frac{\sum_{i=0} y_{i+1} + y_{i+2} + \cdots + y_{i+30}}{n}, \quad n=30$$

In [ ]:
roczne['sr_kroczaca'] = roczne['yi'].rolling(window=30).mean()

# linia trendu dla zakresu danych
roczne['trend'] = a * roczne['ti'] + b

print(roczne[['rok', 'yi', 'sr_kroczaca', 'trend']].to_string(index=False))

## 5. Wykres – śr. roczna / śr. krocząca 30-letnia / linia trendu

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(roczne['rok'], roczne['yi'],
        color='steelblue', lw=1.2, label='Śr. roczna')

ax.plot(roczne['rok'], roczne['sr_kroczaca'],
        color='red', lw=2, label='Śr. krocząca 30-letnia')

ax.plot(roczne['rok'], roczne['trend'],
        color='black', lw=2.5, label=f'Śr. linia trendu  (a = {a*10:.3f} °C/10 lat)')

ax.set_xlabel('Czas [Rok]', fontsize=11)
ax.set_ylabel('Temperatura [°C]', fontsize=11)
ax.set_title(f'Analiza trendu temperatury – SULEJÓW ({rok_start}–{roczne["rok"].iloc[-1]})', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('trend_sulejow.png', dpi=150)
plt.show()

## 6. Podsumowanie i dyskusja

### 6.1 Współczynnik kierunkowy trendu liniowego

Obliczony współczynnik $a$ oznacza zmianę średniej rocznej temperatury na jednostkę czasu.
Jeśli $a > 0$ → trend rosnący; jeśli $a < 0$ → trend malejący.

Prognozowana temperatura na rok 2100 wyznaczona jest z prostej regresji i zakłada kontynuację obserwowanego trendu liniowego. Należy jednak pamiętać, że trend liniowy jest uproszczeniem – w rzeczywistości zmiany klimatu mogą mieć charakter nieliniowy.

Dla okresu **1960–1990** linia trendu może przeszacowywać lub niedoszacowywać rzeczywistych temperatur w zależności od tego, jak kształtowały się wartości roczne w tym przedziale względem prostej regresji.

### 6.2 Średnia krocząca 30-letnia

Średnia krocząca wygładza krótkoterminowe wahania i uwidacznia długookresową tendencję.
- Gdy linia rocznych temperatur (niebieska) leży **powyżej** średniej kroczącej (czerwonej) → lokalny **trend wzrostowy**.
- Gdy linia rocznych temperatur leży **poniżej** średniej kroczącej → lokalny **trend spadkowy**.

### 6.3 Porównanie obu metod

| Metoda | Zalety | Wady |
|---|---|---|
| Trend liniowy | prosta interpretacja, prognoza | zakłada stałą szybkość zmian |
| Średnia krocząca 30-letnia | eliminuje anomalie, zgodna ze standardem WMO | brak wartości dla pierwszych 29 lat, brak prognozy |